# JupyterLite で学ぶ SimPy 入門：離散事象シミュレーション

このノートブックは、**JupyterLite（ブラウザだけで動く Jupyter 環境）** 上で、
Python の離散事象シミュレーション（DES）ライブラリ **SimPy** の基礎を一から学ぶためのチュートリアルです。

## 対象者
- Python の基本（関数、for 文、リスト）を理解している方
- 待ち行列（レジ、窓口、コールセンター）や在庫管理など「時間とともに起きる出来事」をシミュレーションしたい方
- SimPy を初めて使う方

## このチュートリアルで学ぶこと
1. 離散事象シミュレーションとは
2. Environment とプロセス（ジェネレータ関数）
3. Resource と待ち行列（銀行の窓口）
4. 優先度つきの資源（PriorityResource）
5. 在庫モデル（Container と Store）
6. イベントの同期と割り込み
7. シミュレーション実験の進め方（乱数の種、繰り返し、信頼区間）
8. 経済への応用（コールセンターの人員計画、在庫コストの最小化）

## 使い方
- セルを上から順に `Shift + Enter` で実行してください。
- 各章の最後に **練習問題** があります。「解答欄」に自分でコードを書いてから「解答例」を開いて確認しましょう。

## 0. 環境準備（JupyterLite 用）

まず、必要なライブラリをインストールします。

In [ ]:
# JupyterLite 用のパッケージインストール
try:
    import piplite
    await piplite.install(["simpy", "numpy", "pandas", "matplotlib", "japanize-matplotlib-jlite"])
except ImportError:
    pass

In [ ]:
import random
import simpy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import japanize_matplotlib_jlite  # 日本語表示用（plt の後に import する）

plt.rcParams["figure.figsize"] = (8, 4.5)
print(f"SimPy バージョン: {simpy.__version__}")

---
## 1. 離散事象シミュレーションとは

**離散事象シミュレーション（Discrete Event Simulation, DES）** は、「客が到着した」「サービスが終わった」
「在庫が届いた」といった **出来事（イベント）** が起きる時刻だけを追いかけて、システムの動きを再現する手法です。
時間を 1 秒ずつ進めるのではなく、次のイベントまで一気にジャンプするので効率的です。

| 用語 | 意味 | 例 |
|---|---|---|
| イベント | ある時刻に起きる出来事 | 客の到着、サービス完了 |
| プロセス | 時間の経過とともに進む一連の行動 | 客が来て、並んで、サービスを受けて、帰る |
| 資源（Resource） | 同時に使える数に上限があるもの | 窓口、レジ、電話回線 |
| 待ち行列 | 資源が空くのを待つ列 | レジの行列 |

SimPy では、プロセスを **ジェネレータ関数**（`yield` を使う関数）として書き、
`Environment` がイベントの順番どおりに実行してくれます。

---
## 2. Environment とプロセス

### 2.1 最初のプロセス

- `env = simpy.Environment()`：シミュレーションの「世界」を作る
- `yield env.timeout(5)`：5 時間単位だけ待つ（その間に他のプロセスが動く）
- `env.now`：現在のシミュレーション時刻
- `env.process(...)`：プロセスを登録する
- `env.run(until=...)`：指定時刻まで実行する

In [ ]:
def clock(env):
    """5 単位時間ごとに時刻を表示するプロセス"""
    while True:
        print(f"時刻 {env.now}: チクタク")
        yield env.timeout(5)


env = simpy.Environment()
env.process(clock(env))
env.run(until=20)

### 2.2 複数のプロセスを同時に動かす

プロセスは何個でも登録できます。`env.run()` はイベントを時刻順に処理するので、
複数のプロセスが **並行して** 動いているように見えます。

In [ ]:
def customer(env, name, arrive_after, service_time):
    yield env.timeout(arrive_after)
    print(f"{env.now:5.1f}: {name} が到着")
    yield env.timeout(service_time)
    print(f"{env.now:5.1f}: {name} のサービス完了")


env = simpy.Environment()
env.process(customer(env, "客A", arrive_after=0, service_time=4))
env.process(customer(env, "客B", arrive_after=1, service_time=2))
env.process(customer(env, "客C", arrive_after=2, service_time=3))
env.run()

上の例では窓口の数に制限がないので、3 人は待たずに同時にサービスを受けています。
「同時に使える数の上限」を表すのが次章の **Resource** です。

### 2.3 プロセスの戻り値と `env.process` の待ち合わせ

プロセスは `return` で値を返せます。別のプロセスを `yield env.process(...)` で待つこともできます。

In [ ]:
def make_coffee(env, name):
    yield env.timeout(3)
    return f"{name} のコーヒー"


def barista(env):
    result = yield env.process(make_coffee(env, "太郎"))   # 終わるまで待って戻り値を受け取る
    print(f"{env.now}: {result} ができました")


env = simpy.Environment()
env.process(barista(env))
env.run()

### 2.4 timeout に値を持たせる・時刻は小数でもよい

`env.timeout(時間, value=...)` とすると、待ち終わったときにその値を受け取れます。時間は小数でも構いません。

In [ ]:
def delivery(env):
    parcel = yield env.timeout(2.5, value="荷物 A")
    print(f"{env.now}: {parcel} が届いた")
    parcel = yield env.timeout(1.25, value="荷物 B")
    print(f"{env.now}: {parcel} が届いた")


env = simpy.Environment()
env.process(delivery(env))
env.run()
print("シミュレーション終了時刻:", env.now)

### 練習問題 1

1. 「時刻 0 に開店、10 単位時間ごとに『在庫を確認』と表示する」プロセス `stock_check` を作り、40 まで実行してください。
2. 3 台の機械 `machine1`〜`machine3` が、それぞれ 2, 3, 4 単位時間ごとに「部品を 1 個生産」と表示するプロセスを同時に動かし、時刻 12 までに合計何個生産されたかを数えてください（リストに記録してから `len` で数える）。

In [ ]:
# 練習問題 1 の解答欄：ここにコードを書いてください

<details>
<summary><strong>練習問題 1 の解答例を見る</strong></summary>

```python
# 1
def stock_check(env):
    while True:
        print(f"時刻 {env.now}: 在庫を確認")
        yield env.timeout(10)


env = simpy.Environment()
env.process(stock_check(env))
env.run(until=40)

# 2
produced = []


def machine(env, name, cycle):
    while True:
        yield env.timeout(cycle)
        produced.append((env.now, name))


env = simpy.Environment()
for name, cycle in [("machine1", 2), ("machine2", 3), ("machine3", 4)]:
    env.process(machine(env, name, cycle))
env.run(until=12)
print("生産数:", len(produced))
```

</details>

---
## 3. Resource と待ち行列（銀行の窓口）

**`simpy.Resource(env, capacity=n)`** は「同時に n 人まで使える資源」です。
使うときは `with resource.request() as req: yield req` と書き、`with` ブロックを抜けると自動で返却されます。

### 3.1 窓口が 1 つの銀行

In [ ]:
def bank_customer(env, name, counter, service_time):
    arrive = env.now
    print(f"{env.now:5.1f}: {name} が到着（待ち人数 {len(counter.queue)}）")
    with counter.request() as req:        # 窓口を要求
        yield req                          # 空くまで待つ
        wait = env.now - arrive
        print(f"{env.now:5.1f}: {name} のサービス開始（待ち時間 {wait:.1f}）")
        yield env.timeout(service_time)    # サービスを受ける
    print(f"{env.now:5.1f}: {name} が退店")


env = simpy.Environment()
counter = simpy.Resource(env, capacity=1)
env.process(bank_customer(env, "客A", counter, service_time=4))
env.process(bank_customer(env, "客B", counter, service_time=2))
env.process(bank_customer(env, "客C", counter, service_time=3))
env.run()

### 3.2 ランダムに到着する客と待ち時間の記録

現実の客は不規則に到着します。到着間隔とサービス時間を **指数分布** の乱数で作り、
待ち時間をリストに記録します。乱数は `random.Random(seed)` を使って再現できるようにします。

In [ ]:
def customer_generator(env, counter, rng, arrival_rate, service_rate, wait_times):
    """客を次々に生成するプロセス"""
    i = 0
    while True:
        yield env.timeout(rng.expovariate(arrival_rate))     # 次の客が来るまで待つ
        i += 1
        env.process(bank_customer_quiet(env, f"客{i}", counter, rng.expovariate(service_rate), wait_times))


def bank_customer_quiet(env, name, counter, service_time, wait_times):
    """表示せずに待ち時間だけ記録する客"""
    arrive = env.now
    with counter.request() as req:
        yield req
        wait_times.append(env.now - arrive)
        yield env.timeout(service_time)


def simulate_bank(n_counters, arrival_rate=1.0, service_rate=1.2, sim_time=500, seed=0):
    rng = random.Random(seed)
    env = simpy.Environment()
    counter = simpy.Resource(env, capacity=n_counters)
    wait_times = []
    env.process(customer_generator(env, counter, rng, arrival_rate, service_rate, wait_times))
    env.run(until=sim_time)
    return wait_times


waits = simulate_bank(n_counters=1)
print("客数:", len(waits))
print(f"平均待ち時間: {np.mean(waits):.2f}  最大待ち時間: {np.max(waits):.2f}")
print(f"待たずに済んだ割合: {np.mean(np.array(waits) == 0):.1%}")

### 3.3 待ち時間の分布

In [ ]:
plt.hist(waits, bins=30, edgecolor="black")
plt.title("待ち時間の分布（窓口 1 つ）")
plt.xlabel("待ち時間")
plt.ylabel("客数")
plt.show()

### 3.4 窓口の数を変えて比べる

In [ ]:
results = {}
for n in [1, 2, 3]:
    w = simulate_bank(n_counters=n, seed=0)
    results[n] = np.mean(w)
    print(f"窓口 {n} つ: 平均待ち時間 {np.mean(w):.2f}, 最大 {np.max(w):.2f}")

plt.bar([str(k) for k in results], list(results.values()))
plt.title("窓口の数と平均待ち時間")
plt.xlabel("窓口の数")
plt.ylabel("平均待ち時間")
plt.show()

### 3.5 待ち行列の長さの推移を記録する

一定間隔で `len(counter.queue)` を記録する **監視プロセス** を追加すると、行列の長さの時間変化が分かります。

In [ ]:
def monitor(env, counter, log, interval=1.0):
    while True:
        log.append((env.now, len(counter.queue), counter.count))   # (時刻, 待ち人数, 使用中の窓口数)
        yield env.timeout(interval)


rng = random.Random(1)
env = simpy.Environment()
counter = simpy.Resource(env, capacity=1)
wait_times, log = [], []
env.process(customer_generator(env, counter, rng, 1.0, 1.2, wait_times))
env.process(monitor(env, counter, log))
env.run(until=200)

log_df = pd.DataFrame(log, columns=["時刻", "待ち人数", "使用中"])
log_df.plot(x="時刻", y="待ち人数")
plt.title("待ち行列の長さの推移（窓口 1 つ）")
plt.ylabel("待ち人数")
plt.show()
print("平均待ち人数:", round(log_df["待ち人数"].mean(), 2), " 窓口の稼働率:", round(log_df["使用中"].mean(), 2))

### 3.6 理論値（M/M/1 モデル）と比べる

到着間隔とサービス時間が指数分布で窓口が 1 つの待ち行列は **M/M/1 モデル** と呼ばれ、
平均待ち時間の理論値が分かっています（λ = 到着率、μ = サービス率、ρ = λ/μ）。

$$W_q = \frac{\rho}{\mu - \lambda}$$

シミュレーションの結果がこれに近づくか確かめましょう（長く回すほど近づきます）。

In [ ]:
lam, mu = 1.0, 1.2
rho = lam / mu
theory = rho / (mu - lam)
print(f"利用率 ρ = {rho:.3f}, 理論上の平均待ち時間 W_q = {theory:.3f}")

for sim_time in [500, 5000, 20000]:
    w = simulate_bank(1, arrival_rate=lam, service_rate=mu, sim_time=sim_time, seed=0)
    print(f"シミュレーション時間 {sim_time:6d}: 平均待ち時間 {np.mean(w):.3f}（客数 {len(w)}）")

### 練習問題 2

1. `simulate_bank` で到着率を 1.0 のまま、サービス率を 1.1, 1.5, 2.0 に変えて、平均待ち時間を比べてください（窓口 1 つ、`seed=0`）。
2. スーパーのレジを想定し、到着率 2.0（1 単位時間に平均 2 人）、サービス率 1.2 のとき、平均待ち時間を 1 以下にするには最低何台のレジが必要か、`n_counters` を 1 から順に増やして調べてください。

In [ ]:
# 練習問題 2 の解答欄：ここにコードを書いてください

<details>
<summary><strong>練習問題 2 の解答例を見る</strong></summary>

```python
# 1
for mu in [1.1, 1.5, 2.0]:
    w = simulate_bank(1, arrival_rate=1.0, service_rate=mu, seed=0)
    print(f"サービス率 {mu}: 平均待ち時間 {np.mean(w):.2f}")

# 2
for n in range(1, 8):
    w = simulate_bank(n, arrival_rate=2.0, service_rate=1.2, seed=0)
    print(f"レジ {n} 台: 平均待ち時間 {np.mean(w):.2f}")
    if np.mean(w) <= 1:
        print("→ 必要なレジの台数:", n)
        break
```

</details>

---
## 4. 優先度つきの資源：PriorityResource

**`simpy.PriorityResource`** を使うと、`request(priority=数値)` で優先順位を付けられます（**小さい数ほど優先**）。
ここでは VIP 客（優先度 0）と一般客（優先度 1）が同じ窓口を使う状況を再現します。

In [ ]:
def priority_customer(env, counter, rng, priority, service_rate, record):
    arrive = env.now
    with counter.request(priority=priority) as req:
        yield req
        record.append({"種別": "VIP" if priority == 0 else "一般", "待ち時間": env.now - arrive})
        yield env.timeout(rng.expovariate(service_rate))


def priority_generator(env, counter, rng, record, vip_share=0.2):
    while True:
        yield env.timeout(rng.expovariate(1.0))
        priority = 0 if rng.random() < vip_share else 1
        env.process(priority_customer(env, counter, rng, priority, 1.2, record))


rng = random.Random(3)
env = simpy.Environment()
counter = simpy.PriorityResource(env, capacity=1)
record = []
env.process(priority_generator(env, counter, rng, record))
env.run(until=500)

rec_df = pd.DataFrame(record)
print(rec_df.groupby("種別")["待ち時間"].agg(["count", "mean", "max"]).round(2))

In [ ]:
rec_df.boxplot(column="待ち時間", by="種別")
plt.suptitle("")
plt.title("種別ごとの待ち時間")
plt.ylabel("待ち時間")
plt.show()

### 練習問題 3

1. VIP の割合 `vip_share` を 0.2 から 0.5 に増やすと、一般客の平均待ち時間はどう変わりますか。同じ `seed=3` で比べてください。
2. 優先度を 3 段階（0: 緊急、1: 通常、2: 低）にした病院の受付を作り、各段階の平均待ち時間を表示してください（割合はそれぞれ 10%、60%、30% とします）。

In [ ]:
# 練習問題 3 の解答欄：ここにコードを書いてください

<details>
<summary><strong>練習問題 3 の解答例を見る</strong></summary>

```python
# 1
for share in [0.2, 0.5]:
    rng = random.Random(3)
    env = simpy.Environment()
    counter = simpy.PriorityResource(env, capacity=1)
    record = []
    env.process(priority_generator(env, counter, rng, record, vip_share=share))
    env.run(until=500)
    d = pd.DataFrame(record)
    print(f"VIP 割合 {share}: 一般客の平均待ち時間 {d[d['種別'] == '一般']['待ち時間'].mean():.2f}")


# 2
def hospital_generator(env, counter, rng, record):
    while True:
        yield env.timeout(rng.expovariate(1.0))
        u = rng.random()
        priority = 0 if u < 0.1 else (1 if u < 0.7 else 2)
        env.process(hospital_patient(env, counter, rng, priority, record))


def hospital_patient(env, counter, rng, priority, record):
    arrive = env.now
    with counter.request(priority=priority) as req:
        yield req
        record.append({"優先度": priority, "待ち時間": env.now - arrive})
        yield env.timeout(rng.expovariate(1.2))


rng = random.Random(3)
env = simpy.Environment()
counter = simpy.PriorityResource(env, capacity=1)
record = []
env.process(hospital_generator(env, counter, rng, record))
env.run(until=500)
print(pd.DataFrame(record).groupby("優先度")["待ち時間"].mean().round(2))
```

</details>

---
## 5. 在庫モデル：Container と Store

- **`simpy.Container`**：数量（連続量）を入れる容器。`put(量)` / `get(量)` で出し入れし、足りなければ待つ
- **`simpy.Store`**：個別の品物（オブジェクト）を入れる倉庫。`put(品物)` / `get()`

### 5.1 発注点方式の在庫管理（Container）

倉庫の在庫を `Container` で表し、次のルールで管理します（**(s, S) 方式**）。

- 毎日ランダムな量の需要が来る。在庫があれば出荷し、足りない分は **欠品** として記録する
- 在庫が発注点 `s` を下回ったら、上限 `S` まで補充する注文を出す。注文は **リードタイム** 後に届く

In [ ]:
def demand_process(env, stock, rng, daily_mean, shortage_log):
    """毎日の需要"""
    while True:
        yield env.timeout(1)
        demand = rng.randint(0, daily_mean * 2)
        available = min(demand, stock.level)
        if available > 0:
            yield stock.get(available)
        shortage_log.append(demand - available)          # 欠品量（0 なら欠品なし）


def order_process(env, stock, s, S, lead_time, order_log):
    """発注点方式の補充"""
    while True:
        yield env.timeout(1)
        if stock.level <= s and not order_process.pending:
            order_process.pending = True
            qty = S - stock.level
            order_log.append((env.now, qty))
            yield env.timeout(lead_time)                  # 納品を待つ
            yield stock.put(qty)
            order_process.pending = False


def inventory_monitor(env, stock, level_log):
    while True:
        level_log.append((env.now, stock.level))
        yield env.timeout(1)


def simulate_inventory(s, S, lead_time=3, days=120, daily_mean=5, seed=0):
    rng = random.Random(seed)
    env = simpy.Environment()
    stock = simpy.Container(env, init=S, capacity=S)
    shortage_log, order_log, level_log = [], [], []
    order_process.pending = False
    env.process(demand_process(env, stock, rng, daily_mean, shortage_log))
    env.process(order_process(env, stock, s, S, lead_time, order_log))
    env.process(inventory_monitor(env, stock, level_log))
    env.run(until=days)
    return pd.DataFrame(level_log, columns=["日", "在庫"]), shortage_log, order_log


level_df, shortages, orders = simulate_inventory(s=15, S=50)
print("発注回数:", len(orders), " 欠品が起きた日数:", sum(1 for x in shortages if x > 0), " 欠品総量:", sum(shortages))

In [ ]:
level_df.plot(x="日", y="在庫", legend=False)
plt.axhline(15, color="red", linestyle="--", label="発注点 s = 15")
plt.axhline(50, color="gray", linestyle=":", label="上限 S = 50")
plt.title("在庫水準の推移（(s, S) = (15, 50)）")
plt.ylabel("在庫")
plt.legend()
plt.show()

### 5.2 Store：品物を 1 個ずつ扱う

`Store` は「個別の品物」を扱います。生産者が製品を `put` し、消費者が `get` で受け取ります。
在庫がなければ消費者は届くまで待ちます。

In [ ]:
def producer(env, store, rng):
    i = 0
    while True:
        yield env.timeout(rng.uniform(1, 3))
        i += 1
        yield store.put(f"製品{i}")
        print(f"{env.now:5.1f}: 製品{i} を生産（在庫 {len(store.items)}）")


def consumer(env, store, name, rng):
    while True:
        yield env.timeout(rng.uniform(2, 4))
        item = yield store.get()
        print(f"{env.now:5.1f}: {name} が {item} を購入")


rng = random.Random(7)
env = simpy.Environment()
store = simpy.Store(env, capacity=5)          # 最大 5 個まで保管
env.process(producer(env, store, rng))
env.process(consumer(env, store, "客X", rng))
env.process(consumer(env, store, "客Y", rng))
env.run(until=15)

### 5.3 容量の上限で待たされる：Container の put

`Container` には容量の上限があります。倉庫がいっぱいのときに `put` すると、空きができるまで **生産側が待たされます**
（`get` で在庫が足りないときに消費側が待つのと対称です）。

In [ ]:
def factory(env, warehouse):
    for i in range(1, 7):
        yield env.timeout(1)
        print(f"{env.now:4.1f}: 製品 {i} を倉庫へ（在庫 {warehouse.level} → 搬入待ち）")
        yield warehouse.put(1)                       # 満杯なら空くまで待つ
        print(f"{env.now:4.1f}: 搬入完了（在庫 {warehouse.level}）")


def truck(env, warehouse):
    while True:
        yield env.timeout(4)
        taken = min(2, warehouse.level)
        if taken > 0:
            yield warehouse.get(taken)
            print(f"{env.now:4.1f}: トラックが {taken} 個出荷（在庫 {warehouse.level}）")


env = simpy.Environment()
warehouse = simpy.Container(env, init=0, capacity=3)
env.process(factory(env, warehouse))
env.process(truck(env, warehouse))
env.run(until=14)

### 練習問題 4

1. `simulate_inventory` で発注点 `s` を 5, 15, 25 に変え（`S=50`）、欠品総量と発注回数を比べてください。
2. リードタイムを 3 日から 7 日に延ばすと、`s=15` のとき欠品はどれくらい増えますか。

In [ ]:
# 練習問題 4 の解答欄：ここにコードを書いてください

<details>
<summary><strong>練習問題 4 の解答例を見る</strong></summary>

```python
# 1
for s in [5, 15, 25]:
    _, sh, od = simulate_inventory(s=s, S=50)
    print(f"s={s}: 欠品総量 {sum(sh)}, 発注回数 {len(od)}")

# 2
for lt in [3, 7]:
    _, sh, od = simulate_inventory(s=15, S=50, lead_time=lt)
    print(f"リードタイム {lt} 日: 欠品総量 {sum(sh)}")
```

</details>

---
## 6. イベントの同期と割り込み

### 6.1 複数のイベントを待つ：AllOf / AnyOf

- `yield env.all_of([...])`（または `e1 & e2`）：**すべて** 終わるまで待つ
- `yield env.any_of([...])`（または `e1 | e2`）：**どれか 1 つ** 終わったら進む

In [ ]:
def parallel_tasks(env):
    task_a = env.timeout(3, value="書類作成")
    task_b = env.timeout(5, value="審査")
    results = yield env.all_of([task_a, task_b])          # 両方終わるまで待つ
    print(f"{env.now}: 両方完了 → {list(results.todict().values())}")

    fast = env.timeout(2, value="バス")
    slow = env.timeout(6, value="電車")
    first = yield env.any_of([fast, slow])                # 先に来た方に乗る
    print(f"{env.now}: 先に来たのは {list(first.todict().values())}")


env = simpy.Environment()
env.process(parallel_tasks(env))
env.run()

### 6.2 割り込み：Interrupt

動いているプロセスに `process.interrupt()` を送ると、そのプロセスの `yield` の場所で
`simpy.Interrupt` 例外が発生します。機械の故障や、客が待ちきれずに帰る状況を表せます。

In [ ]:
def machine_work(env):
    total_done = 0
    while True:
        try:
            yield env.timeout(4)                 # 1 個作るのに 4 単位時間
            total_done += 1
            print(f"{env.now:5.1f}: 製品完成（累計 {total_done}）")
        except simpy.Interrupt as interrupt:
            print(f"{env.now:5.1f}: 故障！（原因: {interrupt.cause}）修理に 3 単位時間")
            yield env.timeout(3)


def breakdown(env, machine_proc, rng):
    while True:
        yield env.timeout(rng.expovariate(1 / 10))     # 平均 10 単位時間ごとに故障
        machine_proc.interrupt("部品の摩耗")


rng = random.Random(5)
env = simpy.Environment()
proc = env.process(machine_work(env))
env.process(breakdown(env, proc, rng))
env.run(until=40)

### 6.3 `&` と `|` で書く

`env.all_of` / `env.any_of` は、演算子 `&`（両方）と `|`（どちらか）でも書けます。

In [ ]:
def two_step_approval(env):
    dept_a = env.timeout(2, value="部門A 承認")
    dept_b = env.timeout(4, value="部門B 承認")
    yield dept_a & dept_b                      # 両部門の承認を待つ
    print(f"{env.now}: 両部門の承認がそろった")

    quick = env.timeout(1, value="速達")
    normal = env.timeout(3, value="普通郵便")
    arrived = yield quick | normal              # 先に届いた方
    print(f"{env.now}: 先に届いたのは {list(arrived.todict().values())[0]}")


env = simpy.Environment()
env.process(two_step_approval(env))
env.run()

### 練習問題 5

1. 「客は窓口を最大 5 単位時間しか待たず、それを過ぎたら帰る」客のプロセスを作ってください（ヒント：`req | env.timeout(5)` を `yield` し、結果に `req` が含まれているかで判定します。`req in result` で確認できます）。到着率 1.0、サービス率 1.2、窓口 1 つ、`seed=0`、200 単位時間で、帰ってしまった客の割合を求めてください。

In [ ]:
# 練習問題 5 の解答欄：ここにコードを書いてください

<details>
<summary><strong>練習問題 5 の解答例を見る</strong></summary>

```python
def impatient_customer(env, counter, rng, service_rate, stats):
    with counter.request() as req:
        result = yield req | env.timeout(5)
        if req in result:
            stats["served"] += 1
            yield env.timeout(rng.expovariate(service_rate))
        else:
            stats["left"] += 1


def impatient_generator(env, counter, rng, stats):
    while True:
        yield env.timeout(rng.expovariate(1.0))
        env.process(impatient_customer(env, counter, rng, 1.2, stats))


rng = random.Random(0)
env = simpy.Environment()
counter = simpy.Resource(env, capacity=1)
stats = {"served": 0, "left": 0}
env.process(impatient_generator(env, counter, rng, stats))
env.run(until=200)
print(stats, " 帰った割合:", round(stats["left"] / (stats["served"] + stats["left"]), 3))
```

</details>

---
## 7. シミュレーション実験の進め方

乱数を使うシミュレーションの結果は **1 回だけでは信用できません**。次の手順が基本です。

1. **乱数の種（seed）を固定** して、結果を再現できるようにする
2. seed を変えて **何回も繰り返し**（レプリケーション）、平均と散らばりを見る
3. 平均の **95% 信頼区間** を計算する： 平均 ± 1.96 × 標準偏差 ÷ √回数（回数が多いとき）
4. 開始直後（空の状態）の影響を除くため、**ウォームアップ期間** のデータを捨てることもある

### 7.1 繰り返し実行と信頼区間

In [ ]:
def replicate(n_counters, n_rep=20, **kwargs):
    means = []
    for seed in range(n_rep):
        w = simulate_bank(n_counters, seed=seed, **kwargs)
        means.append(np.mean(w))
    means = np.array(means)
    half = 1.96 * means.std(ddof=1) / np.sqrt(n_rep)
    return means.mean(), half


for n in [1, 2, 3]:
    m, h = replicate(n)
    print(f"窓口 {n} つ: 平均待ち時間 {m:.2f} ± {h:.2f}（95% 信頼区間）")

In [ ]:
ns = [1, 2, 3, 4]
means, halfs = zip(*[replicate(n) for n in ns])
plt.errorbar(ns, means, yerr=halfs, marker="o", capsize=5)
plt.title("窓口の数と平均待ち時間（20 回の平均と 95% 信頼区間）")
plt.xlabel("窓口の数")
plt.ylabel("平均待ち時間")
plt.xticks(ns)
plt.grid(True)
plt.show()

### 7.2 ウォームアップ期間を除く

最初の一定期間の客は行列が空の状態で来るため、待ち時間が短めに出ます。
到着時刻を記録して、前半を捨ててから集計してみましょう。

In [ ]:
def simulate_bank_with_time(n_counters, arrival_rate=1.0, service_rate=1.2, sim_time=500, seed=0):
    rng = random.Random(seed)
    env = simpy.Environment()
    counter = simpy.Resource(env, capacity=n_counters)
    records = []

    def cust(env):
        arrive = env.now
        with counter.request() as req:
            yield req
            records.append((arrive, env.now - arrive))
            yield env.timeout(rng.expovariate(service_rate))

    def gen(env):
        while True:
            yield env.timeout(rng.expovariate(arrival_rate))
            env.process(cust(env))

    env.process(gen(env))
    env.run(until=sim_time)
    return pd.DataFrame(records, columns=["到着時刻", "待ち時間"])


df = simulate_bank_with_time(1, seed=0)
print("全期間の平均待ち時間      :", round(df["待ち時間"].mean(), 2))
print("ウォームアップ 100 を除く :", round(df[df["到着時刻"] > 100]["待ち時間"].mean(), 2))

### 練習問題 6

1. `replicate` を使って、到着率 1.0・サービス率 1.2・窓口 2 つの平均待ち時間を、繰り返し回数 5 回と 50 回で比べ、信頼区間の幅がどう変わるか確認してください。
2. `simulate_bank_with_time` の結果を使い、到着時刻を 50 ごとの区間に分けて（`pd.cut`）、区間ごとの平均待ち時間を折れ線グラフにしてください。

In [ ]:
# 練習問題 6 の解答欄：ここにコードを書いてください

<details>
<summary><strong>練習問題 6 の解答例を見る</strong></summary>

```python
# 1
for rep in [5, 50]:
    m, h = replicate(2, n_rep=rep)
    print(f"{rep} 回: {m:.3f} ± {h:.3f}")

# 2
df = simulate_bank_with_time(1, seed=0)
df["区間"] = pd.cut(df["到着時刻"], bins=range(0, 501, 50))
by_bin = df.groupby("区間", observed=True)["待ち時間"].mean()
plt.plot(range(len(by_bin)), by_bin.values, marker="o")
plt.xticks(range(len(by_bin)), [str(i) for i in by_bin.index], rotation=45)
plt.xlabel("到着時刻の区間")
plt.ylabel("平均待ち時間")
plt.tight_layout()
plt.show()
```

</details>

---
## 8. 経済への応用

### 8.1 コールセンターの人員計画

オペレーターを増やすと人件費は増えますが、客の待ち時間（機会損失）は減ります。
**総費用 = 人件費 + 待ち時間コスト** が最小になる人数を探します。

- 人件費：1 人あたり 1 単位時間に 20
- 待ち時間コスト：客 1 人の待ち時間 1 単位あたり 15

In [ ]:
def call_center_cost(n_staff, wage=20, wait_cost=15, arrival_rate=3.0, service_rate=1.0, sim_time=300, n_rep=5):
    total_costs = []
    for seed in range(n_rep):
        waits = simulate_bank(n_staff, arrival_rate=arrival_rate, service_rate=service_rate, sim_time=sim_time, seed=seed)
        labor = wage * n_staff * sim_time
        waiting = wait_cost * sum(waits)
        total_costs.append(labor + waiting)
    return np.mean(total_costs)


staff_range = range(3, 11)
costs = [call_center_cost(n) for n in staff_range]
for n, c in zip(staff_range, costs):
    print(f"オペレーター {n:2d} 人: 総費用 {c:10.0f}")
best = staff_range[int(np.argmin(costs))]
print("→ 総費用が最小になる人数:", best)

In [ ]:
plt.plot(list(staff_range), costs, marker="o")
plt.axvline(best, color="red", linestyle="--", label=f"最適 {best} 人")
plt.title("オペレーター数と総費用")
plt.xlabel("オペレーターの人数")
plt.ylabel("総費用")
plt.legend()
plt.grid(True)
plt.show()

### 8.2 在庫コストの最小化

第 5 章の (s, S) 方式について、**保管費用 + 欠品費用 + 発注費用** が最小になる発注点 s を探します。

- 保管費用：在庫 1 個・1 日あたり 1
- 欠品費用：欠品 1 個あたり 20
- 発注費用：1 回あたり 50

In [ ]:
def inventory_cost(s, S=50, holding=1, shortage=20, ordering=50, n_rep=5):
    costs = []
    for seed in range(n_rep):
        level_df, shortages, orders = simulate_inventory(s=s, S=S, seed=seed)
        c = holding * level_df["在庫"].sum() + shortage * sum(shortages) + ordering * len(orders)
        costs.append(c)
    return np.mean(costs)


s_values = range(0, 41, 5)
inv_costs = [inventory_cost(s) for s in s_values]
for s, c in zip(s_values, inv_costs):
    print(f"発注点 s={s:2d}: 平均総費用 {c:8.0f}")
best_s = list(s_values)[int(np.argmin(inv_costs))]
print("→ 最適な発注点:", best_s)

plt.plot(list(s_values), inv_costs, marker="o")
plt.axvline(best_s, color="red", linestyle="--", label=f"最適 s = {best_s}")
plt.title("発注点と在庫関連の総費用")
plt.xlabel("発注点 s")
plt.ylabel("総費用")
plt.legend()
plt.grid(True)
plt.show()

---
## まとめ

| トピック | 主なクラス・関数 |
|---|---|
| 基本 | `simpy.Environment()`, `env.process()`, `env.run(until=)`, `env.now`, `yield env.timeout()` |
| 資源 | `simpy.Resource(env, capacity)`, `with res.request() as req: yield req`, `res.queue`, `res.count` |
| 優先度 | `simpy.PriorityResource`, `request(priority=)` |
| 在庫 | `simpy.Container(init, capacity)`, `get()/put()`, `level`; `simpy.Store`, `items` |
| 同期・割り込み | `env.all_of()`, `env.any_of()`, `e1 \| e2`, `process.interrupt()`, `except simpy.Interrupt` |
| 実験 | `random.Random(seed)`, レプリケーション, 95% 信頼区間, ウォームアップ |
| 応用 | 総費用（人件費 + 待ちコスト、保管 + 欠品 + 発注）の最小化 |

## 次のステップ

- `python/mesa/mesa_beginner_tutorial.ipynb` — エージェント同士の相互作用を扱うエージェントベースモデル
- `python/scipy/scipy_optimize_beginner_tutorial.ipynb` — シミュレーションではなく数式で最適解を求める方法
- 待ち行列理論（M/M/1 モデル）の理論値と、このノートのシミュレーション結果を比べてみましょう

---
## 総合演習：時間帯で混雑が変わるスーパーのレジ計画

1 日（0〜12 時間）の営業で、客の到着率が時間帯によって変わるスーパーを考えます。

- 到着率：0〜4 時は 1.0 人/単位時間、4〜8 時は 3.0（ピーク）、8〜12 時は 1.5
- サービス率：1 台あたり 1.2
- レジは営業中ずっと同じ台数を開ける

課題：

1. 到着率が時間帯で変わる客の生成プロセスを書き、`simulate_store(n_registers, seed)` を作ってください（戻り値は各客の到着時刻と待ち時間の DataFrame）。
2. レジ 3〜7 台について、seed 0〜4 の 5 回ずつ実行し、平均待ち時間と「待ち時間 2 以上の客の割合」を表にしてください。
3. 「平均待ち時間 1 以下」を満たす最小のレジ台数を求め、その台数での時間帯別平均待ち時間を棒グラフにしてください。

In [ ]:
# 総合演習の解答欄：ここにコードを書いてください

### 総合演習の解答例

自分で書いてから、次のセルを実行して結果を比べてみてください。

In [ ]:
def arrival_rate_at(t):
    if t < 4:
        return 1.0
    elif t < 8:
        return 3.0
    return 1.5


def simulate_store(n_registers, seed=0, sim_time=12, service_rate=1.2):
    rng = random.Random(seed)
    env = simpy.Environment()
    registers = simpy.Resource(env, capacity=n_registers)
    records = []

    def shopper(env):
        arrive = env.now
        with registers.request() as req:
            yield req
            records.append((arrive, env.now - arrive))
            yield env.timeout(rng.expovariate(service_rate))

    def gen(env):
        while True:
            yield env.timeout(rng.expovariate(arrival_rate_at(env.now)))
            env.process(shopper(env))

    env.process(gen(env))
    env.run(until=sim_time)
    return pd.DataFrame(records, columns=["到着時刻", "待ち時間"])


# 2. レジ台数ごとの集計
rows = []
for n in range(3, 8):
    frames = [simulate_store(n, seed=s) for s in range(5)]
    all_df = pd.concat(frames)
    rows.append({"レジ台数": n, "平均待ち時間": all_df["待ち時間"].mean(), "待ち 2 以上の割合": (all_df["待ち時間"] >= 2).mean()})
table = pd.DataFrame(rows).set_index("レジ台数").round(3)
print(table)

# 3. 条件を満たす最小台数と時間帯別の待ち時間
best_n = table[table["平均待ち時間"] <= 1].index.min()
print("平均待ち時間 1 以下になる最小台数:", best_n)
best_df = pd.concat([simulate_store(best_n, seed=s) for s in range(5)])
best_df["時間帯"] = pd.cut(best_df["到着時刻"], bins=[0, 4, 8, 12], labels=["0-4", "4-8", "8-12"], include_lowest=True)
by_period = best_df.groupby("時間帯", observed=True)["待ち時間"].mean()
by_period.plot(kind="bar", rot=0)
plt.title(f"レジ {best_n} 台のときの時間帯別平均待ち時間")
plt.xlabel("時間帯")
plt.ylabel("平均待ち時間")
plt.show()

お疲れさまでした！ SimPy を使えば、待ち行列・在庫・生産ラインなど「時間の流れ」がある経済の問題を、
数式では扱いにくい細かなルールも含めて再現できます。身近な行列（学食、券売機、病院）を題材にモデルを作ってみましょう。